In [1]:
pip install torch diffusers transformers accelerate sentencepiece opencv-python

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
import numpy as np
import cv2
from PIL import Image
from diffusers import StableDiffusion3Pipeline
from tqdm import tqdm

# ==========================================
# 1. Cấu hình & Khởi tạo Model
# ==========================================
# Lưu ý: Bạn cần chấp nhận điều khoản của SD3 trên Hugging Face 
# và có thể cần login bằng: huggingface-cli login
model_id = "stabilityai/stable-diffusion-3-medium-diffusers"

print("--- Đang tải mô hình SD3 (vui lòng đợi)... ---")
pipe = StableDiffusion3Pipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,
    text_encoder_3=None, # Tùy chọn: bỏ bớt encoder nếu thiếu VRAM (8GB-12GB)
    tokenizer_3=None
)

# Tối ưu hóa bộ nhớ
pipe.enable_model_cpu_offload() 
# Nếu máy bạn cực mạnh (>24GB VRAM), dùng: pipe.to("cuda")

# ==========================================
# 2. Hàm nội suy (Slerp - Spherical Linear Interpolation)
# ==========================================
# Slerp giúp việc chuyển cảnh mượt mà hơn so với cộng tuyến tính thông thường
def slerp(val, low, high):
    low_norm = low / torch.norm(low)
    high_norm = high / torch.norm(high)
    omega = torch.acos((low_norm * high_norm).sum())
    so = torch.sin(omega)
    res = (torch.sin((1.0 - val) * omega) / so) * low + (torch.sin(val * omega) / so) * high
    return res

# ==========================================
# 3. Logic chính: Random Walk trong Latent Space
# ==========================================
def generate_sd3_random_walk(
    prompt, 
    num_keyframes=5,    # Số lượng điểm dừng chính (ngẫu nhiên)
    steps_per_key=12,   # Số bước chuyển tiếp giữa các điểm (tạo độ mượt)
    guidance_scale=7.0,
    seed=42
):
    torch.manual_seed(seed)
    
    # Kích thước Latent của SD3: (1, 16, 64, 64) cho ảnh 512x512 
    # Hoặc (1, 16, 128, 128) cho ảnh 1024x1024
    latent_shape = (1, 16, 64, 64) 
    
    # Tạo các "Keyframes" - các khối nhiễu ngẫu nhiên làm mốc
    keyframes = [torch.randn(latent_shape, device="cuda", dtype=torch.float16) for _ in range(num_keyframes)]
    
    all_images = []

    print(f"Bắt đầu quá trình tạo chuỗi hình ảnh...")

    for i in range(len(keyframes) - 1):
        start_latent = keyframes[i]
        end_latent = keyframes[i+1]
        
        print(f"Đang chuyển tiếp từ mốc {i} sang mốc {i+1}...")
        
        for step in tqdm(range(steps_per_key)):
            # Tính toán tỷ lệ nội suy (từ 0.0 đến 1.0)
            t = step / steps_per_key
            
            # Thực hiện nội suy để lấy latent trung gian
            interpolated_latent = slerp(t, start_latent, end_latent)
            
            # Render ảnh từ latent
            with torch.no_grad():
                image = pipe(
                    prompt=prompt,
                    latents=interpolated_latent,
                    num_inference_steps=25, # Giảm xuống để chạy nhanh hơn
                    guidance_scale=guidance_scale,
                    output_type="pil"
                ).images[0]
            
            all_images.append(image)

    return all_images

# ==========================================
# 4. Lưu kết quả thành Video
# ==========================================
def export_to_video(images, output_path="sd3_walk.mp4", fps=15):
    print(f"--- Đang đóng gói video: {output_path} ---")
    frame_size = images[0].size # (Width, Height)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, frame_size)
    
    for img in images:
        # Chuyển từ PIL (RGB) sang OpenCV (BGR)
        frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        out.write(frame)
    
    out.release()
    print("Hoàn tất!")

# ==========================================
# 5. Thực thi
# ==========================================
if __name__ == "__main__":
    MY_PROMPT = "A dreamlike underwater city with neon lights, high resolution, digital art style"
    
    # Tạo chuỗi ảnh
    generated_frames = generate_sd3_random_walk(
        prompt=MY_PROMPT,
        num_keyframes=4,    # 4 mốc ngẫu nhiên
        steps_per_key=10    # 10 ảnh trung gian giữa mỗi mốc => Tổng 40 ảnh
    )
    
    # Xuất video
    export_to_video(generated_frames, "dream_walk_sd3.mp4")

--- Đang tải mô hình SD3 (vui lòng đợi)... ---


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Bắt đầu quá trình tạo chuỗi hình ảnh...
Đang chuyển tiếp từ mốc 0 sang mốc 1...


...

  0%|          | 0/25 [00:00<?, ?it/s]

 90%|█████████ | 9/10 [01:39<00:11, 11.05s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 10/10 [01:50<00:00, 11.03s/it]

--- Đang đóng gói video: dream_walk_sd3.mp4 ---
Hoàn tất!


In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
from diffusers import StableDiffusion3Pipeline
from tqdm import tqdm
from huggingface_hub import login

# ==========================================
# 1. Xác thực & Cấu hình Low VRAM
# ==========================================
# Dán mã Token bạn vừa lấy vào đây
HF_TOKEN = "" 
login(token=HF_TOKEN)

model_id = "stabilityai/stable-diffusion-3-medium-diffusers"

print("--- Đang tải mô hình SD3 phiên bản LOW VRAM... ---")
pipe = StableDiffusion3Pipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,
    # Bỏ qua bộ mã hóa T5 (chiếm ~10GB VRAM) để chạy được trên card 8GB
    text_encoder_3=None, 
    tokenizer_3=None,
    use_auth_token=True
)

# Tối ưu hóa bộ nhớ tối đa
pipe.enable_model_cpu_offload() 

# ==========================================
# 2. Hàm nội suy Slerp (Spherical Linear Interpolation)
# ==========================================
def slerp(val, low, high):
    low_norm = low / torch.norm(low)
    high_norm = high / torch.norm(high)
    omega = torch.acos(torch.clamp((low_norm * high_norm).sum(), -1, 1))
    so = torch.sin(omega)
    if so == 0:
        return (1.0 - val) * low + val * high
    res = (torch.sin((1.0 - val) * omega) / so) * low + (torch.sin(val * omega) / so) * high
    return res

# ==========================================
# 3. Logic Random Walk
# ==========================================
def generate_low_vram_walk(prompt, num_keyframes=3, steps_per_key=10):
    latent_shape = (1, 16, 64, 64) # Chuẩn SD3 Medium
    
    # Tạo các mốc Latent ngẫu nhiên
    keyframes = [torch.randn(latent_shape, device="cuda", dtype=torch.float16) for _ in range(num_keyframes)]
    
    all_images = []

    for i in range(len(keyframes) - 1):
        start_latent = keyframes[i]
        end_latent = keyframes[i+1]
        
        print(f"Đang xử lý chuyển tiếp mốc {i+1}/{len(keyframes)-1}...")
        
        for step in tqdm(range(steps_per_key)):
            t = step / steps_per_key
            interpolated_latent = slerp(t, start_latent, end_latent)
            
            with torch.no_grad():
                # Sinh ảnh
                image = pipe(
                    prompt=prompt,
                    latents=interpolated_latent,
                    num_inference_steps=28, 
                    guidance_scale=7.0,
                    output_type="pil"
                ).images[0]
            
            all_images.append(image)

    return all_images

# ==========================================
# 4. Xuất Video
# ==========================================
def save_video(images, filename="output_sd3_low.mp4"):
    frame_size = images[0].size
    out = cv2.VideoWriter(filename, cv2.VideoWriter_fourcc(*'mp4v'), 10, frame_size)
    for img in images:
        out.write(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    out.release()
    print(f"Đã lưu video thành công: {filename}")

# ==========================================
# Chạy chương trình
# ==========================================
if __name__ == "__main__":
    PROMPT = "A cosmic nebula transforming into a digital phoenix, vibrant colors, 2k resolution"
    
    frames = generate_low_vram_walk(PROMPT, num_keyframes=3, steps_per_key=15)
    save_video(frames)

--- Đang tải mô hình SD3 phiên bản LOW VRAM... ---


model_index.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Keyword arguments {'use_auth_token': True} are not expected by StableDiffusion3Pipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

 93%|█████████▎| 14/15 [02:42<00:11, 11.69s/it]

  0%|          | 0/28 [00:00<?, ?it/s]

100%|██████████| 15/15 [02:54<00:00, 11.65s/it]


Đã lưu video thành công: output_sd3_low.mp4
